# GTM Agents Migration — Internal Checkpoint Tests

**Internal QA only — not part of the client-facing lab.**

These are the PASS/FAIL checkpoints that were previously embedded at the end of each
lab notebook. They verify that every part of the demo is wired correctly before it is
presented. Run this notebook top-to-bottom **after** running `gtm-01` through `gtm-04`
(the checks read objects those notebooks create).

Each cell asserts on failure, so a clean run == all parts healthy.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

---
## Checkpoint 0 — Shared Foundation (Part 0)

Source: `gtm-01-foundation.ipynb`

In [ ]:
checks = []

rc = {r['TBL']: r['ROWS'] for r in session.sql("""
    SELECT 'REPS' AS tbl, COUNT(*) AS rows FROM REPS
    UNION ALL SELECT 'EMAILS', COUNT(*) FROM EMAILS
    UNION ALL SELECT 'OUTCOMES', COUNT(*) FROM OUTCOMES
    UNION ALL SELECT 'EMAIL_FRAMEWORK', COUNT(*) FROM EMAIL_FRAMEWORK
""").collect()}
checks.append(('REPS > 0', rc.get('REPS',0) > 0))
checks.append(('EMAILS in 2000-5000', 2000 <= rc.get('EMAILS',0) <= 5000))
checks.append(('OUTCOMES > 0', rc.get('OUTCOMES',0) > 0))
checks.append(('EMAIL_FRAMEWORK > 0', rc.get('EMAIL_FRAMEWORK',0) > 0))

# Cortex Analyst semantic view returns rows
analyst_rows = session.sql("""
    SELECT * FROM SEMANTIC_VIEW(EMAIL_GTM_SV DIMENSIONS reps.region METRICS reply_rate)
""").count()
checks.append(('Analyst semantic view returns rows', analyst_rows > 0))

# Governed tool executes and returns a non-empty array
tool_size = session.sql("SELECT ARRAY_SIZE(GTM_TEAM_PERFORMANCE('')) AS n").collect()[0]['N']
checks.append(('Governed SQL tool returns rows', (tool_size or 0) > 0))

# Cortex Search service is ready
svc = session.sql("SHOW CORTEX SEARCH SERVICES LIKE 'FRAMEWORK_SEARCH' IN SCHEMA GTMAGENTS.DEMO").count()
checks.append(('Cortex Search service exists', svc > 0))

print('CHECKPOINT 0')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — proceed to Part A' if overall else 'FAIL — fix above before Part A')
assert overall, 'Checkpoint 0 failed'

---
## Checkpoint A — BEFORE: Claude Code + MCP (Part A)

Source: `gtm-02-before-mcp.ipynb`

In [ ]:
checks = []

# MCP server exists and lists the expected tools
desc = session.sql("DESCRIBE MCP SERVER GTMAGENTS_MCP").collect()
spec = ''.join(str(r.as_dict()) for r in desc)
for t in ['gtm-analyst', 'framework-search', 'gtm_team_performance']:
    checks.append((f"MCP tool '{t}' present", t in spec))

# OAuth integration ENABLED + secrets retrievable
session.sql("USE ROLE ACCOUNTADMIN").collect()
integ = session.sql("SHOW SECURITY INTEGRATIONS LIKE 'GTMAGENTS_MCP_OAUTH'").collect()
checks.append(('OAuth integration exists', len(integ) > 0))
checks.append(('OAuth integration ENABLED', len(integ) > 0 and integ[0]['enabled']))
try:
    sec = session.sql("SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('GTMAGENTS_MCP_OAUTH') AS s").collect()[0]['S']
    checks.append(('OAuth secrets retrievable', 'OAUTH_CLIENT_ID' in sec))
except Exception as e:
    checks.append(('OAuth secrets retrievable', False))
session.sql("USE ROLE SYSADMIN").collect()

# DEFAULT_ROLE / DEFAULT_WAREHOUSE set on the MCP user (warn-level: non-null)
me = session.sql("SELECT CURRENT_USER() AS u").collect()[0]['U']
session.sql("USE ROLE ACCOUNTADMIN").collect()
urow = session.sql(f"SHOW USERS LIKE '{me}'").collect()[0].as_dict()
session.sql("USE ROLE SYSADMIN").collect()
checks.append(('MCP user has DEFAULT_ROLE', bool(urow.get('default_role'))))
checks.append(('MCP user has DEFAULT_WAREHOUSE', bool(urow.get('default_warehouse'))))

# In-plane tool executes (real, measurable) — the Snowflake-side work the MCP brain would call.
# (We no longer assert a fabricated MCP latency row exists; MCP cost/latency is external and not measured here.)
try:
    session.sql("SELECT GTM_TEAM_PERFORMANCE('AMER')").collect()
    checks.append(('In-plane tool GTM_TEAM_PERFORMANCE executes', True))
except Exception:
    checks.append(('In-plane tool GTM_TEAM_PERFORMANCE executes', False))

print('CHECKPOINT A')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
# DEFAULT_ROLE/WH are warn-level (needed only for the live Claude connect), not hard blockers
hard = [ok for name, ok in checks if 'DEFAULT_' not in name]
print()
print('RESULT:', 'PASS — proceed to Part B' if all(hard) else 'FAIL — fix above before Part B')
if not all(ok for _, ok in checks):
    print('NOTE: set DEFAULT_ROLE/DEFAULT_WAREHOUSE before the live Claude connect (Section 4).')

---
## Checkpoint B — AFTER: multi-agent Cortex Agents (Part B)

Source: `gtm-03-after-agents.ipynb`

In [ ]:
checks = []

# Each specialist reachable via its agent-to-agent wrapper procedure
for proc, q in [('RUN_SCORING_AGENT', 'Score email 2'),
                ('RUN_RECOMMENDATION_AGENT', 'Which quality tier has the highest win rate?'),
                ('RUN_COACHING_AGENT', 'Rewrite this: just checking in, let me know if interested.')]:
    try:
        r = session.sql(f"CALL {proc}('{q}')").collect()[0][0]
        checks.append((f'{proc} responded', bool(r) and len(str(r)) > 20))
    except Exception as e:
        checks.append((f'{proc} responded', False))

# Supervisor exists with a budget cap in its spec
desc = session.sql("DESCRIBE AGENT GTM_SUPERVISOR").collect()
spec = ''.join(str(r.as_dict()) for r in desc)
checks.append(('Supervisor has budget cap', 'budget' in spec))
checks.append(('Supervisor binds 3 specialists', all(t in spec for t in ['ScoringSpecialist','RecommendationSpecialist','CoachingSpecialist'])))

# Routing was logged (scoring escalation decisions)
rl = session.sql('SELECT COUNT(*) AS n FROM ROUTING_LOG').collect()[0]['N']
checks.append(('ROUTING_LOG has decisions', rl > 0))

# AI_FILTER gate reduced processed volume vs analyze-everything
cc = {r['APPROACH']: r['EMAILS_TREATED'] for r in session.sql('SELECT approach, emails_treated FROM FILTER_VOLUME').collect()}
checks.append(('Targeted gate < analyze-all', cc.get('targeted_filter', 1e9) < cc.get('analyze_all', 0)))

print('CHECKPOINT B')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — proceed to Part C' if overall else 'FAIL — fix above before Part C')
assert overall, 'Checkpoint B failed'

---
## Checkpoint C — Native agent evaluation (Part C)

Source: `gtm-04-evals.ipynb`

In [ ]:
checks = []
def n(sql):
    return session.sql(sql).collect()[0][0]

# Both runs produced scored rows
for run in ['baseline-v1','improved-v2']:
    cnt = n(f"SELECT COUNT(*) FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA('GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','{run}'))")
    checks.append((f'{run} has scored rows', cnt > 0))

# Routing metric present and non-null
tsa = n("SELECT COUNT(*) FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA('GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1')) WHERE METRIC_NAME='tool_selection_accuracy' AND EVAL_AGG_SCORE IS NOT NULL")
checks.append(('tool_selection_accuracy present & non-null', tsa > 0))

# Dataset covers single_tool AND refusal
procs = n("SELECT COUNT(DISTINCT ground_truth:process::string) FROM AGENT_EVAL_QUESTIONS WHERE ground_truth:process::string IN ('single_tool','refusal')")
checks.append(('dataset covers single_tool + refusal', procs == 2))

# History has >= 2 runs
runs = n('SELECT COUNT(DISTINCT run_name) FROM EVAL_SCORE_HISTORY')
checks.append(('EVAL_SCORE_HISTORY has >= 2 runs', runs >= 2))

# production alias assigned
vers = session.sql('SHOW VERSIONS IN AGENT GTM_SUPERVISOR').collect()
has_prod = any('production' in str(r.as_dict()).lower() for r in vers)
checks.append(('production alias assigned', has_prod))

print('CHECKPOINT C')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — evaluation complete; the lab is fully validated' if overall else 'FAIL — fix above')
assert overall, 'Checkpoint C failed'